In [ ]:
!pip install autogen-agentchat==0.2.38

In [ ]:
!pip install ag2[openai,interop-langchain]

In [11]:
import autogen
from autogen import ConversableAgent, UserProxyAgent, AssistantAgent
from autogen import register_function
from langchain_community.tools import TavilySearchResults
from autogen.interop import Interoperability
import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display

In [12]:
import os
from getpass import getpass

In [13]:
OPENAI_KEY = getpass('Enter Open AI API Key: ')
os.environ["OPENAI_API_KEY"] = OPENAI_KEY

Enter Open AI API Key: ··········


In [24]:
llm_config = {
    "config_list": [{"model": "gpt-4o-mini", "api_key": os.environ["OPENAI_API_KEY"]}],
}

In [25]:
bmi_tool = ConversableAgent(
    name="bmi_calculator",
    system_message="""You are a helpful AI assistant specialized in health metrics calculations.
                    When a user provides health information, carefully extract the following parameters:
                    - weight: numeric value in kilograms (extract the number, ignore units like 'kg')
                    - height: numeric value in centimeters (extract the number, ignore units like 'cm')
                    - age: numeric value in years (extract the number)
                    - gender: string value ('male' or 'female', extract from the message)

                    Example: If user says "height=186cm, weight=98kg, age=30, gender=male", extract:
                    weight=98, height=186, age=30, gender="male"

                    Always call the health_metrics function with the extracted numeric values and gender string.
                    After presenting the results, end your message with 'TERMINATE'.""",
    llm_config=llm_config,
    is_termination_msg=lambda msg: msg.get("content") is not None and "TERMINATE" in msg["content"]
)

In [26]:
user_proxy = UserProxyAgent(
    name="User",
    llm_config=False,
    is_termination_msg=lambda msg: msg.get("content") is not None and "TERMINATE" in msg["content"],
    human_input_mode="TERMINATE",
    code_execution_config=False
)

In [27]:
def health_metrics(weight=None, height=None, age=None, gender=None):
    """
    Calculates health metrics from basic physical details.

    Inputs:
        weight (kg): Person's weight in kilograms.
        height (cm): Person's height in centimeters.
        age (years): Person's age in years.
        gender (str): 'male' or 'female'.

    Outputs:
        A dictionary containing:
            BMI: Body Mass Index (float)
            BMI_Category: Underweight / Normal / Overweight / Obese
            Body_Fat_Percentage: Estimated body fat % (float)

    Purpose:
        Use this function to evaluate a user's basic health indicators.
        Helpful for fitness apps, chat agents, and health assessment tools.
    """
    # Validate required parameters
    if weight is None or height is None or age is None or gender is None:
        return {"error": "Missing required parameters. Please provide weight, height, age, and gender."}

    # Convert weight and height to appropriate types
    try:
        weight = float(weight)
        height = float(height)
        age = int(age)
        gender = str(gender).lower()
    except (ValueError, TypeError) as e:
        return {"error": f"Invalid parameter types: {str(e)}"}

    # Convert height to meters
    height_m = height / 100

    # --- BMI ---
    bmi = weight / (height_m ** 2)

    # --- BMI Category ---
    if bmi < 18.5:
        category = "Underweight"
    elif bmi < 25:
        category = "Normal"
    elif bmi < 30:
        category = "Overweight"
    else:
        category = "Obese"

    # --- Body Fat % (Deurenberg formula) ---
    # gender: male = 1, female = 0
    gender_factor = 1 if gender.lower() == "male" else 0

    body_fat = (1.20 * bmi) + (0.23 * age) - (10.8 * gender_factor) - 5.4

    return {
        "BMI": round(bmi, 2),
        "BMI_Category": category,
        "Body_Fat_Percentage": round(body_fat, 2)
    }

In [28]:
# Register the function using autogen's register_function utility
# This registers it so bmi_tool knows about the function and user_proxy can execute it
register_function(
    health_metrics,
    caller=bmi_tool,
    executor=user_proxy,
    description="""Calculates health metrics (BMI, BMI category, body fat percentage) from basic physical details.
Required parameters:
- weight: numeric value in kilograms (float or int)
- height: numeric value in centimeters (float or int)
- age: numeric value in years (int)
- gender: string value, either "male" or "female"

Example usage: health_metrics(weight=98, height=186, age=30, gender="male")"""
)

In [29]:
# Also register for execution on bmi_tool itself (in case it needs to execute directly)
bmi_tool.register_for_execution()(health_metrics)

# Diet Planner

In [30]:
# Diet Planner Agent
diet_planner = ConversableAgent(
    name="diet_planner",
    system_message="""You are a professional nutritionist and meal planning expert.
                    You create personalized one-week meal plans based on:
                    - Health metrics (BMI, BMI Category, Body Fat Percentage) from previous BMI calculation
                    - User demographics (age, gender)
                    - Diet preferences (veg, vegan, non-veg)

                    When you receive information:
                    1. Extract BMI, BMI Category, and Body Fat Percentage from the previous conversation summary
                    2. Use the provided user information (age, gender, height, weight, diet preference)
                    3. Analyze the BMI category to determine caloric needs
                    4. Consider age and gender for nutritional requirements
                    5. Respect the diet preference (veg/vegan/non-veg)
                    6. Create a balanced, nutritious one-week meal plan

                    Always call the generate_meal_plan function with all the extracted and provided information.
                    Extract numeric values from the previous summary (e.g., "BMI: 28.33" -> bmi=28.33).
                    After presenting the meal plan, end your message with 'TERMINATE'.""",
    llm_config=llm_config,
    is_termination_msg=lambda msg: msg.get("content") is not None and "TERMINATE" in msg["content"]
)

In [31]:
def generate_meal_plan(bmi=None, bmi_category=None, body_fat_percentage=None,
                       age=None, gender=None, diet_preference=None,
                       height=None, weight=None):
    """
    Generates a personalized one-week meal plan based on health metrics and preferences.

    Inputs:
        bmi (float): Body Mass Index value
        bmi_category (str): BMI category (Underweight/Normal/Overweight/Obese)
        body_fat_percentage (float): Body fat percentage
        age (int): Person's age in years
        gender (str): 'male' or 'female'
        diet_preference (str): 'veg', 'vegan', or 'non-veg'
        height (float): Height in centimeters (optional, for caloric calculation)
        weight (float): Weight in kilograms (optional, for caloric calculation)

    Outputs:
        A dictionary containing:
            weekly_meal_plan: Detailed meal plan for 7 days
            daily_calories: Recommended daily caloric intake
            macronutrient_breakdown: Protein, carbs, fats distribution
            recommendations: Personalized dietary recommendations

    Purpose:
        Create a comprehensive meal plan tailored to individual health status and preferences.
    """
    # Validate required parameters
    if bmi is None or bmi_category is None or age is None or gender is None or diet_preference is None:
        return {"error": "Missing required parameters. Please provide BMI, BMI category, age, gender, and diet preference."}

    # Convert inputs
    try:
        bmi = float(bmi)
        age = int(age)
        gender = str(gender).lower()
        diet_preference = str(diet_preference).lower()
        bmi_category = str(bmi_category)
    except (ValueError, TypeError) as e:
        return {"error": f"Invalid parameter types: {str(e)}"}

    # Calculate daily caloric needs based on BMI category and gender
    # Base metabolic rate approximation
    if weight and height:
        # Using Mifflin-St Jeor Equation approximation
        if gender == "male":
            bmr = 10 * weight + 6.25 * height - 5 * age + 5
        else:
            bmr = 10 * weight + 6.25 * height - 5 * age - 161
    else:
        # Fallback: estimate based on BMI category
        if bmi_category.lower() == "underweight":
            base_cal = 2000 if gender == "male" else 1800
        elif bmi_category.lower() == "normal":
            base_cal = 2200 if gender == "male" else 2000
        elif bmi_category.lower() == "overweight":
            base_cal = 1800 if gender == "male" else 1600
        else:  # Obese
            base_cal = 1600 if gender == "male" else 1400
        bmr = base_cal

    # Adjust calories based on BMI category
    if bmi_category.lower() == "underweight":
        daily_calories = int(bmr * 1.2)  # Increase for weight gain
        goal = "weight gain"
    elif bmi_category.lower() == "normal":
        daily_calories = int(bmr * 1.0)  # Maintenance
        goal = "maintenance"
    elif bmi_category.lower() == "overweight":
        daily_calories = int(bmr * 0.85)  # Moderate deficit
        goal = "weight loss"
    else:  # Obese
        daily_calories = int(bmr * 0.75)  # Higher deficit
        goal = "weight loss"

    # Macronutrient breakdown (percentage)
    if goal == "weight loss":
        protein_pct = 30
        carbs_pct = 40
        fats_pct = 30
    elif goal == "weight gain":
        protein_pct = 25
        carbs_pct = 50
        fats_pct = 25
    else:  # maintenance
        protein_pct = 25
        carbs_pct = 45
        fats_pct = 30

    # Generate meal plan based on diet preference
    days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

    if diet_preference == "vegan":
        protein_sources = ["tofu", "lentils", "chickpeas", "quinoa", "tempeh", "black beans", "edamame"]
        meal_options = {
            "breakfast": ["Oatmeal with berries", "Smoothie bowl", "Avocado toast", "Chia pudding", "Tofu scramble"],
            "lunch": ["Lentil curry with rice", "Chickpea salad", "Quinoa bowl", "Vegan wrap", "Stir-fried vegetables"],
            "dinner": ["Tofu stir-fry", "Vegan pasta", "Stuffed bell peppers", "Vegan curry", "Roasted vegetables"],
            "snacks": ["Nuts", "Fruits", "Hummus with veggies", "Vegan protein bar"]
        }
    elif diet_preference == "veg":
        protein_sources = ["paneer", "lentils", "chickpeas", "yogurt", "milk", "cheese", "eggs"]
        meal_options = {
            "breakfast": ["Scrambled eggs with toast", "Oatmeal with milk", "Paneer paratha", "Yogurt with fruits", "Vegetable omelet"],
            "lunch": ["Dal rice", "Paneer curry with roti", "Vegetable biryani", "Rajma rice", "Mixed vegetable curry"],
            "dinner": ["Paneer tikka", "Vegetable pulao", "Stuffed paratha", "Vegetable soup", "Aloo gobi with roti"],
            "snacks": ["Yogurt", "Fruits", "Nuts", "Cheese"]
        }
    else:  # non-veg
        protein_sources = ["chicken", "fish", "eggs", "turkey", "lean beef", "prawns"]
        meal_options = {
            "breakfast": ["Scrambled eggs with toast", "Chicken sandwich", "Omelet with vegetables", "Greek yogurt with fruits", "Protein smoothie"],
            "lunch": ["Grilled chicken with rice", "Fish curry", "Chicken salad", "Beef stir-fry", "Prawn curry"],
            "dinner": ["Baked fish", "Chicken tikka", "Lean beef steak", "Turkey meatballs", "Grilled chicken"],
            "snacks": ["Hard-boiled eggs", "Nuts", "Greek yogurt", "Protein shake"]
        }

    # Generate weekly meal plan
    weekly_plan = {}
    for day in days:
        weekly_plan[day] = {
            "breakfast": meal_options["breakfast"][hash(day + "breakfast") % len(meal_options["breakfast"])],
            "lunch": meal_options["lunch"][hash(day + "lunch") % len(meal_options["lunch"])],
            "dinner": meal_options["dinner"][hash(day + "dinner") % len(meal_options["dinner"])],
            "snack": meal_options["snacks"][hash(day + "snack") % len(meal_options["snacks"])]
        }

    # Generate recommendations
    recommendations = []
    if bmi_category.lower() == "overweight" or bmi_category.lower() == "obese":
        recommendations.append("Focus on portion control and calorie deficit")
        recommendations.append("Include more fiber-rich foods for satiety")
        recommendations.append("Stay hydrated - drink 8-10 glasses of water daily")
    elif bmi_category.lower() == "underweight":
        recommendations.append("Include nutrient-dense foods")
        recommendations.append("Eat frequent, smaller meals")
        recommendations.append("Focus on healthy fats and proteins")
    else:
        recommendations.append("Maintain balanced nutrition")
        recommendations.append("Stay active with regular exercise")

    recommendations.append(f"Target daily calories: {daily_calories} kcal")
    recommendations.append(f"Primary goal: {goal}")

    return {
        "weekly_meal_plan": weekly_plan,
        "daily_calories": daily_calories,
        "macronutrient_breakdown": {
            "protein": f"{protein_pct}%",
            "carbohydrates": f"{carbs_pct}%",
            "fats": f"{fats_pct}%"
        },
        "recommendations": recommendations,
        "diet_preference": diet_preference,
        "bmi_category": bmi_category
    }


# Register the meal plan function

In [32]:

register_function(
    generate_meal_plan,
    caller=diet_planner,
    executor=user_proxy,
    description="""Generates a personalized one-week meal plan based on health metrics and preferences.
Required parameters:
- bmi: Body Mass Index value (float)
- bmi_category: BMI category - "Underweight", "Normal", "Overweight", or "Obese" (string)
- body_fat_percentage: Body fat percentage (float, optional)
- age: Person's age in years (int)
- gender: "male" or "female" (string)
- diet_preference: "veg", "vegan", or "non-veg" (string)
- height: Height in centimeters (float, optional)
- weight: Weight in kilograms (float, optional)

Example usage: generate_meal_plan(bmi=28.33, bmi_category="Overweight", age=30, gender="male", diet_preference="non-veg")"""
)

diet_planner.register_for_execution()(generate_meal_plan)

# Workout Scheduler Agent
workout_scheduler = ConversableAgent(
    name="workout_scheduler",
    system_message="""You are a professional fitness trainer and workout planning expert.
                    You create personalized one-week workout schedules based on:
                    - Meal plan suggestions and recommendations from the Diet Planner Agent
                    - User demographics (age, gender)
                    - Fitness goals aligned with health metrics

                    When you receive information:
                    1. Review the meal plan and recommendations from the previous conversation summary
                    2. Consider the user's age and gender for appropriate exercise intensity
                    3. Create a balanced weekly workout schedule that complements the diet plan
                    4. Include variety: cardio, strength training, flexibility, and rest days
                    5. Adjust intensity based on age and fitness level

                    Always call the generate_workout_plan function with all the provided information.
                    Extract relevant information from the previous conversation summary (BMI category, goals, etc.).
                    After presenting the workout plan, end your message with 'TERMINATE'.""",
    llm_config=llm_config,
    is_termination_msg=lambda msg: msg.get("content") is not None and "TERMINATE" in msg["content"]
)

# Generate WorkoutPlan

In [33]:
def generate_workout_plan(age=None, gender=None, bmi_category=None, fitness_goal=None,
                          daily_calories=None, current_fitness_level="beginner"):
    """
    Generates a personalized one-week workout plan based on user information and health goals.

    Inputs:
        age (int): Person's age in years
        gender (str): 'male' or 'female'
        bmi_category (str): BMI category (Underweight/Normal/Overweight/Obese) - optional
        fitness_goal (str): Primary goal - "weight_loss", "weight_gain", "muscle_building", "maintenance", "general_fitness"
        daily_calories (int): Daily caloric intake from meal plan - optional
        current_fitness_level (str): "beginner", "intermediate", or "advanced" - defaults to "beginner"

    Outputs:
        A dictionary containing:
            weekly_workout_plan: Detailed workout schedule for 7 days
            workout_intensity: Recommended intensity level
            weekly_schedule_summary: Overview of the week
            recommendations: Personalized fitness recommendations

    Purpose:
        Create a comprehensive workout plan tailored to individual needs and goals.
    """
    # Validate required parameters
    if age is None or gender is None:
        return {"error": "Missing required parameters. Please provide age and gender."}

    # Convert inputs
    try:
        age = int(age)
        gender = str(gender).lower()
        if bmi_category:
            bmi_category = str(bmi_category).lower()
        if fitness_goal:
            fitness_goal = str(fitness_goal).lower()
        current_fitness_level = str(current_fitness_level).lower()
    except (ValueError, TypeError) as e:
        return {"error": f"Invalid parameter types: {str(e)}"}

    # Determine fitness goal from BMI category if not provided
    if not fitness_goal:
        if bmi_category == "underweight":
            fitness_goal = "muscle_building"
        elif bmi_category in ["overweight", "obese"]:
            fitness_goal = "weight_loss"
        else:
            fitness_goal = "general_fitness"

    # Adjust intensity based on age
    if age < 25:
        base_intensity = "moderate_to_high"
        recovery_days = 1
    elif age < 40:
        base_intensity = "moderate"
        recovery_days = 2
    elif age < 55:
        base_intensity = "moderate_to_low"
        recovery_days = 2
    else:
        base_intensity = "low_to_moderate"
        recovery_days = 3

    # Adjust based on fitness level
    if current_fitness_level == "advanced":
        intensity_multiplier = 1.2
    elif current_fitness_level == "intermediate":
        intensity_multiplier = 1.0
    else:  # beginner
        intensity_multiplier = 0.8

    # Define workout types based on goal
    if fitness_goal == "weight_loss":
        primary_focus = "cardio"
        secondary_focus = "strength"
        workout_types = {
            "cardio": ["Running", "Cycling", "HIIT", "Swimming", "Rowing", "Elliptical", "Jump Rope"],
            "strength": ["Full Body Circuit", "Upper Body", "Lower Body", "Core Focus"],
            "flexibility": ["Yoga", "Stretching", "Pilates"]
        }
    elif fitness_goal == "muscle_building":
        primary_focus = "strength"
        secondary_focus = "cardio"
        workout_types = {
            "strength": ["Push Day (Chest/Triceps)", "Pull Day (Back/Biceps)", "Leg Day", "Full Body"],
            "cardio": ["Light Jogging", "Walking", "Cycling"],
            "flexibility": ["Yoga", "Stretching", "Mobility Work"]
        }
    elif fitness_goal == "weight_gain":
        primary_focus = "strength"
        secondary_focus = "cardio"
        workout_types = {
            "strength": ["Push Day", "Pull Day", "Leg Day", "Full Body"],
            "cardio": ["Light Cardio", "Walking"],
            "flexibility": ["Stretching", "Yoga"]
        }
    else:  # general_fitness or maintenance
        primary_focus = "balanced"
        workout_types = {
            "cardio": ["Running", "Cycling", "Swimming", "Dancing"],
            "strength": ["Full Body", "Upper Body", "Lower Body", "Core"],
            "flexibility": ["Yoga", "Stretching", "Pilates"]
        }

    # Generate weekly workout plan
    days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    weekly_plan = {}

    # Distribute workouts throughout the week
    workout_count = 0
    for i, day in enumerate(days):
        # Rest days
        if i % (7 - recovery_days) == 0 and workout_count > 0:
            weekly_plan[day] = {
                "type": "Rest Day",
                "activities": ["Light walking or stretching", "Recovery", "Rest"],
                "duration": "30-45 minutes (optional light activity)",
                "intensity": "Very Low"
            }
        else:
            # Alternate workout types
            if workout_count % 3 == 0:
                workout_type = "strength"
                activity = workout_types["strength"][workout_count % len(workout_types["strength"])]
            elif workout_count % 3 == 1:
                workout_type = "cardio"
                activity = workout_types["cardio"][workout_count % len(workout_types["cardio"])]
            else:
                workout_type = "flexibility"
                activity = workout_types["flexibility"][workout_count % len(workout_types["flexibility"])]

            # Determine duration based on workout type and intensity
            if workout_type == "cardio":
                duration = "30-45 minutes" if base_intensity == "moderate_to_high" else "20-30 minutes"
            elif workout_type == "strength":
                duration = "45-60 minutes" if base_intensity == "moderate_to_high" else "30-45 minutes"
            else:  # flexibility
                duration = "20-30 minutes"

            weekly_plan[day] = {
                "type": workout_type.capitalize(),
                "activities": [activity],
                "duration": duration,
                "intensity": base_intensity.replace("_", " ").title()
            }
            workout_count += 1

    # Generate recommendations
    recommendations = []

    if fitness_goal == "weight_loss":
        recommendations.append("Focus on cardio exercises 4-5 times per week")
        recommendations.append("Include strength training 2-3 times per week to preserve muscle mass")
        recommendations.append("Aim for 150+ minutes of moderate-intensity cardio weekly")
    elif fitness_goal == "muscle_building":
        recommendations.append("Prioritize strength training 4-5 times per week")
        recommendations.append("Allow 48 hours rest between training the same muscle groups")
        recommendations.append("Keep cardio light to moderate to support recovery")
    elif fitness_goal == "weight_gain":
        recommendations.append("Focus on progressive strength training")
        recommendations.append("Limit cardio to avoid excessive calorie burn")
        recommendations.append("Ensure adequate rest and recovery")
    else:
        recommendations.append("Maintain a balanced mix of cardio, strength, and flexibility")
        recommendations.append("Aim for at least 150 minutes of moderate activity per week")

    # Age-specific recommendations
    if age >= 40:
        recommendations.append("Include proper warm-up and cool-down (10-15 minutes each)")
        recommendations.append("Focus on joint mobility and flexibility")
        recommendations.append("Listen to your body and adjust intensity as needed")

    # Gender-specific considerations
    if gender == "female":
        recommendations.append("Include weight-bearing exercises for bone health")

    recommendations.append(f"Recommended workout intensity: {base_intensity.replace('_', ' ').title()}")
    recommendations.append(f"Rest days: {recovery_days} per week")

    return {
        "weekly_workout_plan": weekly_plan,
        "workout_intensity": base_intensity.replace("_", " ").title(),
        "fitness_goal": fitness_goal.replace("_", " ").title(),
        "weekly_schedule_summary": {
            "total_workout_days": 7 - recovery_days,
            "rest_days": recovery_days,
            "primary_focus": primary_focus.replace("_", " ").title()
        },
        "recommendations": recommendations,
        "age": age,
        "gender": gender
    }

# Register the workout plan function

In [34]:

register_function(
    generate_workout_plan,
    caller=workout_scheduler,
    executor=user_proxy,
    description="""Generates a personalized one-week workout plan based on user information and fitness goals.
Required parameters:
- age: Person's age in years (int)
- gender: "male" or "female" (string)

Optional parameters:
- bmi_category: BMI category - "Underweight", "Normal", "Overweight", or "Obese" (string)
- fitness_goal: "weight_loss", "weight_gain", "muscle_building", "maintenance", or "general_fitness" (string)
- daily_calories: Daily caloric intake (int)
- current_fitness_level: "beginner", "intermediate", or "advanced" (string, defaults to "beginner")

Example usage: generate_workout_plan(age=30, gender="male", bmi_category="Overweight", fitness_goal="weight_loss")"""
)

workout_scheduler.register_for_execution()(generate_workout_plan)

# User inputs

In [35]:
user_height = 186  # cm
user_weight = 98   # kg
user_age = 30
user_gender = "male"
diet_preference = "non-veg"  # User input: "veg", "vegan", or "non-veg"

In [36]:
# Messages for sequential chat pattern
messages = [
    f"""Calculate BMI height={user_height}cm, weight={user_weight}kg, age={user_age}, gender={user_gender}""",
    f"""Based on the BMI calculation results from the previous conversation summary, please create a personalized one-week meal plan.

User Information:
- Age: {user_age}
- Gender: {user_gender}
- Height: {user_height}cm
- Weight: {user_weight}kg
- Diet Preference: {diet_preference}

Please extract the BMI, BMI Category, and Body Fat Percentage from the previous conversation summary and use the generate_meal_plan function with all the information.""",
    f"""Based on the meal plan and recommendations from the Diet Planner (previous conversation summary), please create a personalized one-week workout schedule.

User Information:
- Age: {user_age}
- Gender: {user_gender}

Please extract the BMI Category, fitness goals, and any relevant recommendations from the previous conversation summaries and use the generate_workout_plan function with all the information."""
]

In [37]:
# Sequential pattern: BMI calculation → Diet Planner → Workout Scheduler
chat_results = autogen.initiate_chats(
    [
        {
            "sender": user_proxy,
            "recipient": bmi_tool,
            "message": messages[0],
            "clear_history": True,
            "summary_method": "last_msg",
            "max_turns": 5
        },
        {
            "sender": user_proxy,
            "recipient": diet_planner,
            "message": messages[1],
            "summary_method": "last_msg",
            "max_turns": 5
        },
        {
            "sender": user_proxy,
            "recipient": workout_scheduler,
            "message": messages[2],
            "summary_method": "last_msg",
            "max_turns": 5
        }
    ]
)


********************************************************************************
Starting a new chat....

********************************************************************************
User (to bmi_calculator):

Calculate BMI height=186cm, weight=98kg, age=30, gender=male

--------------------------------------------------------------------------------

>>>>>>>> USING AUTO REPLY...
bmi_calculator (to User):

***** Suggested tool call (call_dZAoWBIBy82i4uRlvzWdcR0E): health_metrics *****
Arguments: 
{"weight":98,"height":186,"age":30,"gender":"male"}
*******************************************************************************

--------------------------------------------------------------------------------

>>>>>>>> USING AUTO REPLY...

>>>>>>>> EXECUTING FUNCTION health_metrics...
Call ID: call_dZAoWBIBy82i4uRlvzWdcR0E
Input arguments: {'weight': 98, 'height': 186, 'age': 30, 'gender': 'male'}

>>>>>>>> EXECUTED FUNCTION health_metrics...
Call ID: call_dZAoWBIBy82i4uRlvzWdcR0E
In

In [42]:
from IPython.display import display, Markdown

output_md = f"""
============================================================
# Smart Health Assistant - AutoGen
============================================================

### 🧮 Chat 1 → BMI Calculation
**Result:** {chat_results[0].summary if hasattr(chat_results[0], 'summary') else "Completed"}

### 🥗 Chat 2 → Diet Plan
**Result:** {chat_results[1].summary if hasattr(chat_results[1], 'summary') else "Completed"}

### 🏋️ Chat 3 → Workout Schedule
**Result:** {chat_results[2].summary if hasattr(chat_results[2], 'summary') else "Completed"}

============================================================
**Final Output:**
Complete Health Plan with BMI insights, tailored diet, and fitness schedule
============================================================
"""

display(Markdown(output_md))



============================================================
# Smart Health Assistant - AutoGen
============================================================

### 🧮 Chat 1 → BMI Calculation
**Result:** Your health metrics have been calculated:

- BMI: 28.33
- BMI Category: Overweight
- Body Fat Percentage: 24.69%



### 🥗 Chat 2 → Diet Plan
**Result:** Here's your personalized one-week meal plan based on your BMI, body fat percentage, and dietary preferences:

### Daily Caloric Target: 1697 kcal
### Macronutrient Breakdown:
- Protein: 30%
- Carbohydrates: 40%
- Fats: 30%

#### Weekly Meal Plan

**Monday**
- Breakfast: Protein smoothie
- Lunch: Beef stir-fry
- Dinner: Lean beef steak
- Snack: Protein shake

**Tuesday**
- Breakfast: Chicken sandwich
- Lunch: Beef stir-fry
- Dinner: Turkey meatballs
- Snack: Greek yogurt

**Wednesday**
- Breakfast: Chicken sandwich
- Lunch: Beef stir-fry
- Dinner: Lean beef steak
- Snack: Nuts

**Thursday**
- Breakfast: Omelet with vegetables
- Lunch: Grilled chicken with rice
- Dinner: Turkey meatballs
- Snack: Nuts

**Friday**
- Breakfast: Protein smoothie
- Lunch: Beef stir-fry
- Dinner: Baked fish
- Snack: Hard-boiled eggs

**Saturday**
- Breakfast: Omelet with vegetables
- Lunch: Fish curry
- Dinner: Baked fish
- Snack: Protein shake

**Sunday**
- Breakfast: Protein smoothie
- Lunch: Chicken salad
- Dinner: Grilled chicken
- Snack: Protein shake

### Recommendations:
- Focus on portion control and calorie deficit.
- Include more fiber-rich foods for satiety.
- Stay hydrated - drink 8-10 glasses of water daily.
- Primary goal: weight loss.

Feel free to adjust the portions or ingredients based on availability and personal preferences. Enjoy your meals!



### 🏋️ Chat 3 → Workout Schedule
**Result:** Here’s your personalized one-week workout schedule that aligns with your meal plan and supports your weight loss goals:

### Weekly Workout Schedule

**Monday: Strength Training**
- **Activities**: Full Body Circuit
- **Duration**: 30-45 minutes
- **Intensity**: Moderate

**Tuesday: Cardio**
- **Activities**: Cycling
- **Duration**: 20-30 minutes
- **Intensity**: Moderate

**Wednesday: Flexibility**
- **Activities**: Pilates
- **Duration**: 20-30 minutes
- **Intensity**: Moderate

**Thursday: Strength Training**
- **Activities**: Core Focus
- **Duration**: 30-45 minutes
- **Intensity**: Moderate

**Friday: Cardio**
- **Activities**: Rowing
- **Duration**: 20-30 minutes
- **Intensity**: Moderate

**Saturday: Rest Day**
- **Activities**: Light walking or stretching, Recovery
- **Duration**: 30-45 minutes (optional light activity)
- **Intensity**: Very Low

**Sunday: Flexibility**
- **Activities**: Pilates
- **Duration**: 20-30 minutes
- **Intensity**: Moderate

### Summary of Recommendations:
- Focus on cardio exercises 4-5 times per week.
- Include strength training 2-3 times per week to preserve muscle mass.
- Aim for a total of 150+ minutes of moderate-intensity cardio weekly.
- Prioritize moderate workout intensity for effective weight loss.
- Enjoy your rest days to aid recovery and remain active with light activities as needed.

Feel free to adjust the workouts based on your preferences and availability. Good luck with your fitness journey!



============================================================
**Final Output:**  
Complete Health Plan with BMI insights, tailored diet, and fitness schedule  
============================================================


## Example 2

In [43]:
user_height = 178  # cm
user_weight = 75   # kg
user_age = 29
user_gender = "female"
diet_preference = "veg"  # User input: "veg", "vegan", or "non-veg"

# Messages for sequential chat pattern
messages = [
    f"""Calculate BMI height={user_height}cm, weight={user_weight}kg, age={user_age}, gender={user_gender}""",
    f"""Based on the BMI calculation results from the previous conversation summary, please create a personalized one-week meal plan.

User Information:
- Age: {user_age}
- Gender: {user_gender}
- Height: {user_height}cm
- Weight: {user_weight}kg
- Diet Preference: {diet_preference}

Please extract the BMI, BMI Category, and Body Fat Percentage from the previous conversation summary and use the generate_meal_plan function with all the information.""",
    f"""Based on the meal plan and recommendations from the Diet Planner (previous conversation summary), please create a personalized one-week workout schedule.

User Information:
- Age: {user_age}
- Gender: {user_gender}

Please extract the BMI Category, fitness goals, and any relevant recommendations from the previous conversation summaries and use the generate_workout_plan function with all the information."""
]

# Sequential pattern: BMI calculation → Diet Planner → Workout Scheduler
chat_results = autogen.initiate_chats(
    [
        {
            "sender": user_proxy,
            "recipient": bmi_tool,
            "message": messages[0],
            "clear_history": True,
            "summary_method": "last_msg",
            "max_turns": 5
        },
        {
            "sender": user_proxy,
            "recipient": diet_planner,
            "message": messages[1],
            "summary_method": "last_msg",
            "max_turns": 5
        },
        {
            "sender": user_proxy,
            "recipient": workout_scheduler,
            "message": messages[2],
            "summary_method": "last_msg",
            "max_turns": 5
        }
    ]
)


********************************************************************************
Starting a new chat....

********************************************************************************
User (to bmi_calculator):

Calculate BMI height=178cm, weight=75kg, age=29, gender=female

--------------------------------------------------------------------------------

>>>>>>>> USING AUTO REPLY...
bmi_calculator (to User):

***** Suggested tool call (call_SIEUMceoC2OlFbWKIPBUMt5H): health_metrics *****
Arguments: 
{"weight": 75, "height": 178, "age": 29, "gender": "female"}
*******************************************************************************

--------------------------------------------------------------------------------

>>>>>>>> USING AUTO REPLY...

>>>>>>>> EXECUTING FUNCTION health_metrics...
Call ID: call_SIEUMceoC2OlFbWKIPBUMt5H
Input arguments: {'weight': 75, 'height': 178, 'age': 29, 'gender': 'female'}

>>>>>>>> EXECUTED FUNCTION health_metrics...
Call ID: call_SIEUMceoC2OlFb

In [44]:
from IPython.display import display, Markdown

output_md = f"""
============================================================
# Smart Health Assistant - AutoGen
============================================================

### 🧮 Chat 1 → BMI Calculation
**Result:** {chat_results[0].summary if hasattr(chat_results[0], 'summary') else "Completed"}

### 🥗 Chat 2 → Diet Plan
**Result:** {chat_results[1].summary if hasattr(chat_results[1], 'summary') else "Completed"}

### 🏋️ Chat 3 → Workout Schedule
**Result:** {chat_results[2].summary if hasattr(chat_results[2], 'summary') else "Completed"}

============================================================
**Final Output:**
Complete Health Plan with BMI insights, tailored diet, and fitness schedule
============================================================
"""

display(Markdown(output_md))



============================================================
# Smart Health Assistant - AutoGen
============================================================

### 🧮 Chat 1 → BMI Calculation
**Result:** Your calculated health metrics are as follows:
- BMI: 23.67
- BMI Category: Normal
- Body Fat Percentage: 29.68%



### 🥗 Chat 2 → Diet Plan
**Result:** Here is your personalized one-week meal plan based on your health metrics and dietary preferences:

### Weekly Meal Plan

#### Monday
- **Breakfast:** Vegetable omelet
- **Lunch:** Rajma rice
- **Dinner:** Stuffed paratha
- **Snack:** Cheese

#### Tuesday
- **Breakfast:** Oatmeal with milk
- **Lunch:** Rajma rice
- **Dinner:** Vegetable soup
- **Snack:** Nuts

#### Wednesday
- **Breakfast:** Oatmeal with milk
- **Lunch:** Rajma rice
- **Dinner:** Stuffed paratha
- **Snack:** Fruits

#### Thursday
- **Breakfast:** Paneer paratha
- **Lunch:** Dal rice
- **Dinner:** Vegetable soup
- **Snack:** Fruits

#### Friday
- **Breakfast:** Vegetable omelet
- **Lunch:** Rajma rice
- **Dinner:** Paneer tikka
- **Snack:** Yogurt

#### Saturday
- **Breakfast:** Paneer paratha
- **Lunch:** Paneer curry with roti
- **Dinner:** Paneer tikka
- **Snack:** Cheese

#### Sunday
- **Breakfast:** Vegetable omelet
- **Lunch:** Vegetable biryani
- **Dinner:** Aloo gobi with roti
- **Snack:** Cheese

### Nutritional Information
- **Daily Calories:** 1556 kcal
- **Macronutrient Breakdown:** 
  - Protein: 25%
  - Carbohydrates: 45%
  - Fats: 30%

### Recommendations
- Maintain balanced nutrition.
- Stay active with regular exercise.
- Primary goal: maintenance.

Feel free to modify any meal as per your taste preferences. Enjoy your meals!



### 🏋️ Chat 3 → Workout Schedule
**Result:** Here’s your personalized one-week workout schedule designed to complement your meal plan and support your maintenance fitness goal:

### Weekly Workout Plan

#### Monday
- **Type:** Strength
- **Activities:** Full Body
- **Duration:** 30-45 minutes
- **Intensity:** Moderate

#### Tuesday
- **Type:** Cardio
- **Activities:** Cycling
- **Duration:** 20-30 minutes
- **Intensity:** Moderate

#### Wednesday
- **Type:** Flexibility
- **Activities:** Pilates
- **Duration:** 20-30 minutes
- **Intensity:** Moderate

#### Thursday
- **Type:** Strength
- **Activities:** Core
- **Duration:** 30-45 minutes
- **Intensity:** Moderate

#### Friday
- **Type:** Cardio
- **Activities:** Running
- **Duration:** 20-30 minutes
- **Intensity:** Moderate

#### Saturday
- **Type:** Rest Day
- **Activities:** Light walking or stretching, Recovery, Rest
- **Duration:** 30-45 minutes (optional light activity)
- **Intensity:** Very Low

#### Sunday
- **Type:** Flexibility
- **Activities:** Pilates
- **Duration:** 20-30 minutes
- **Intensity:** Moderate

### Recommendations
- Maintain a balanced mix of cardio, strength, and flexibility.
- Aim for at least 150 minutes of moderate activity per week.
- Include weight-bearing exercises for bone health.
- Rest days: 2 per week.

Enjoy your workouts and the positive impact they will have on your fitness journey!



============================================================
**Final Output:**  
Complete Health Plan with BMI insights, tailored diet, and fitness schedule  
============================================================
